In [3]:
# !pip install -r requirements.txt

In [1]:
import pandas as pd
from tqdm.auto import tqdm
import json
from itertools import islice

from river import metrics, dummy, forest, tree, stats, ensemble, drift
from ensemble import DriftAdaptiveEnsemble
from transformer import FeatureDistrict
from utils import warsaw_stream, airline_stream, taxi_stream

seed = 17

with open("transformer/config_dataset.json", "r", encoding="utf-8-sig") as f:
    config = json.load(f)

In [2]:
DATASET = 'Warsaw'
MAX_SAMPLES = 10_000

In [3]:
df_warsaw=pd.read_csv("dataset/warsaw_synthetic.csv")
# df_airplanes=pd.read_csv("dataset/Airplanes_modified.csv")
# df_taxi=pd.read_csv("dataset/taxi_dataset_ordered.csv")

if DATASET=="Airplanes":
    rows = df_airplanes.to_dict(orient='records')
    data = airline_stream(rows)
elif DATASET=="Warsaw":
    rows = df_warsaw.to_dict(orient='records')
    data = warsaw_stream(rows)
elif DATASET=="Taxi":
    rows = df_taxi.to_dict(orient='records')
    data = taxi_stream(rows)
else:
    raise Exception("Dataset ERROR")

cfg = config[DATASET]

In [4]:
models = {
    'MEAN': dummy.StatisticRegressor(stats.Mean()),
    'HTR': tree.HoeffdingTreeRegressor(),
    "ARF": forest.ARFRegressor(n_models=10, seed=seed),
    "ARF_drift": forest.ARFRegressor(n_models=10, seed=seed, drift_detector=drift.ADWIN(), warning_detector=drift.ADWIN()),
    "SRP": ensemble.SRPRegressor(n_models=10, seed=seed),
    "SRP_drift": ensemble.SRPRegressor(n_models=10, seed=seed, drift_detector=drift.ADWIN(), warning_detector=drift.ADWIN()),
    "Ensemble": DriftAdaptiveEnsemble(
            base_estimator=tree.HoeffdingTreeRegressor(),
            drift_detector=drift.ADWIN(), warning_detector=drift.ADWIN(),
            metric=metrics.RMSE(),
            max_ensemble_size=10, 
            retain_initial_model=False
        )
}
transformer = FeatureDistrict(
    dataset=DATASET,
    columns_to_drop=cfg["columns_to_drop"],
)

In [5]:
data = islice(data, MAX_SAMPLES)

In [6]:
x, y = next(data)
print(x)
print(y)

{'timestamp': '2025-01-01 00:09:06', 'start_lat': 52.2532644, 'start_lon': 20.9815095, 'end_lat': 52.159729, 'end_lon': 21.1263282, 'distance_km': 17.78688849267649, 'hour': 0, 'day_of_week': 2, 'is_weekend': 0, 'rush_hour': 0, 'is_holiday': 1, 'pickup_d': 'Left', 'dropoff_d': 'Left', 'weather_clear': 1, 'weather_fog': 0, 'weather_rain': 0, 'weather_snow': 0}
25.13108336695775


In [7]:
rows = []
for i, (x_raw, y) in tqdm(enumerate(data), total=MAX_SAMPLES):
    x = transformer.transform_one(x_raw)
    timestamp = x['timestamp']
    x.pop("timestamp")

    x.pop('pickup_district', None)
    x.pop('dropoff_district', None)
    x.pop('within_district', None)

    row = {
        "i": i,
        "timestamp": timestamp,
        "y_true": y,
    }

    for name, model in models.items():
        pred = model.predict_one(x)
        row[f"y_{name}"] = pred if pred is not None else 0.0
        model.learn_one(x, y)

    rows.append(row)
    
pd.DataFrame(rows).to_csv('results/warsaw.csv')

  0%|          | 0/10000 [00:00<?, ?it/s]